# Where They Should Rank

In [1]:
import numpy as np, pandas as pd
pd.set_option("display.max.columns", 60)
CURRENT_SZN = 2026
POSITIONS = ["QB", "RB", "WR", "TE"]
SCORING = {
    "passing_yards": 0.04,
    "passing_tds": 4,
    "interceptions": -2,
    "rushing_yards": 0.1,
    "rushing_tds": 6,
    "receptions": 1,
    "receiving_yards": 0.1,
    "receiving_tds": 6,
    "two_pt_conversions": 2,
    "fumbles_lost": -2,
    "special_teams_tds": 6,
}

actual_ledger = pd.read_parquet(f"data/ledger_{CURRENT_SZN}.parquet")
weekly = pd.read_parquet(f"data/weekly_points_{CURRENT_SZN}.parquet")
luck = pd.read_parquet(f"data/luck_ledger_{CURRENT_SZN}.parquet")
official = pd.read_parquet(f"data/player_stats_{CURRENT_SZN}.parquet")
official = official[official["season_type"] == "REG"]
players = (official.sort_values("week")
           .groupby("player_id", as_index = False)
           .agg(player_display_name = ("player_display_name", "last"),
                position_group = ("position_group", "last"),
                team = ("team", "last")))

print(f"Actual player-games: {len(weekly):,}\nLuck ledger rows: {len(luck):,}")

Actual player-games: 2,955
Luck ledger rows: 9,552


## Self-inflicted fumbles
Every lost fumble already cost the player 2 points in his actual total, so getting those 2 back goes in the self-inflicted column. The fumbles that replay gave back as luck in 04 are skipped here, so they aren't counted twice.

In [2]:
fumbles = actual_ledger[actual_ledger["stat"] == "fumbles_lost"].copy()
replay_fumbles = luck[(luck["event"] == "replay_reversal") & (luck["stat"] == "fumbles_lost")]
replay_keys = set(zip(replay_fumbles["game_id"], replay_fumbles["play_id"], replay_fumbles["player_id"]))
is_replay = np.array([k in replay_keys for k in zip(fumbles["game_id"], fumbles["game_id"], fumbles["play_id"], fumbles["player_id"])])
fumbles = fumbles[~is_replay]

goal_line = luck[luck["event"] == "down_short_fumble"]
goal_line_keys = set(zip(goal_line["game_id"], goal_line["play_id"], goal_line["player_id"]))
fumbles = fumbles[np.array([k in goal_line_keys for k in zip(fumbles["game_id"], fumbles["play_id"], fumbles["player_id"])], dtype=bool)]
# the fumble cost him points (negative), so the points he should have had are the opposite
fumbles["self_points"] = -fumbles["points"]
print(f"Lost fumbles added to self-inflicted: {len(fumbles):,}\nSkipped (replay assist): {is_replay.sum():,}")

Lost fumbles added to self-inflicted: 5
Skipped (replay assist): 0


## Player-week table
One row per player per game, with actual points, luck, and self-inflicted points side by side. A player can have almost points in a game where he didn't score (a TD wiped out on his only touch), so the table includes every player-game.

In [3]:
GAME = ["season", "week", "game_id", "player_id"]

actual_pg = weekly[GAME + ["fps"]].rename(columns = {"fps": "actual"})
luck_pg = (luck[luck["category"] == "luck"].groupby(GAME, as_index = False)["points"].sum()
           .rename(columns = {"points": "luck"}))
self_pg = pd.concat([
    luck[luck["category"] == "self_inflicted"][GAME + ["points"]].rename(columns = {"points": "self_inflicted"}),
    fumbles[GAME + ["self_points"]].rename(columns = {"self_points": "self_inflicted"}),
]).groupby(GAME, as_index = False)["self_inflicted"].sum()

pg = (actual_pg.merge(luck_pg, on = GAME, how = "outer")
      .merge(self_pg, on = GAME, how = "outer")
      .fillna({"actual": 0, "luck": 0, "self_inflicted": 0}))
pg = pg.merge(players, on = "player_id", how = "left")
pg = pg[pg["position_group"].isin(POSITIONS)]

pg["luck_adjusted"] = pg["actual"] + pg["luck"]
pg["should_be"] = pg["luck_adjusted"] + pg["self_inflicted"]
print(f"Player-games: {len(pg):,}")
pg.sort_values("luck", ascending = False).head(10)

Player-games: 3,096


,season,week,game_id,player_id,actual,luck,self_inflicted,player_display_name,position_group,team,luck_adjusted,should_be
2922,2026,11,2026_11_JAX_NYG,00-0036309,0.00,26.90,0.00,Darnell Mooney,WR,NYG,26.90,26.90
2136,2026,8,2026_08_CLE_PIT,00-0041037,9.70,26.60,0.00,Denzel Boston,WR,CLE,36.30,36.30
1830,2026,7,2026_07_DEN_ARI,00-0033119,16.48,26.48,0.00,Jacoby Brissett,QB,BAL,42.96,42.96
1808,2026,7,2026_07_CLE_TEN,00-0041037,5.70,25.60,1.50,Denzel Boston,WR,CLE,31.30,32.80
1450,2026,5,2026_05_NYG_WAS,00-0039337,29.10,24.20,0.60,Malik Nabers,WR,NYG,53.30,53.90
740,2026,3,2026_03_KC_MIA,00-0039880,11.10,23.20,0.00,Malik Washington,WR,MIA,34.30,34.30
984,2026,4,2026_04_DEN_SF,00-0031408,12.60,22.70,0.00,Mike Evans,WR,SF,35.30,35.30
1915,2026,7,2026_07_LA_LV,00-0039738,17.10,22.70,0.00,Blake Corum,RB,LA,39.80,39.80
1907,2026,7,2026_07_LA_LV,00-0037291,3.40,22.20,0.00,Jalen Nailor,WR,LV,25.60,25.60
1605,2026,6,2026_06_DAL_GB,00-0036264,20.24,22.00,5.32,Jordan Love,QB,GB,42.24,47.56


## Breakdown by event
For each player-game, the points behind each kind of almost point, plus how many plays it happened on.

In [4]:
EVENTS = ["drop", "open_miss", "dpi", "penalty_nullified", "penalty_truncated", "overturned_td",
          "replay_reversal", "down_short", "down_short_fumble", "td_transfer", "hail_mary", "fumble"]

events = pd.concat([
    luck[GAME + ["play_id", "event", "points"]],
    fumbles[GAME + ["play_id", "self_points"]].rename(columns = {"self_points": "points"}).assign(event = "fumble"),
])
ev_points = events.pivot_table(index = GAME, columns = "event", values = "points", aggfunc = "sum", fill_value = 0)
ev_plays = (events.drop_duplicates(GAME + ["play_id", "event"])
            .pivot_table(index = GAME, columns = "event", values = "play_id", aggfunc = "count", fill_value = 0))
ev_points = ev_points.reindex(columns = EVENTS, fill_value = 0).add_suffix("_pts")
ev_plays = ev_plays.reindex(columns = EVENTS, fill_value = 0).add_suffix("_plays")
BREAKDOWN = list(ev_points.columns) + list(ev_plays.columns)

pg = pg.merge(ev_points.join(ev_plays).reset_index(), on = GAME, how = "left")
pg[BREAKDOWN] = pg[BREAKDOWN].fillna(0)
print(f"Player-games: {len(pg):,}")
pg.sort_values("luck", ascending = False).head(10)

Player-games: 3,096


,season,week,game_id,player_id,actual,luck,self_inflicted,player_display_name,position_group,team,luck_adjusted,should_be,drop_pts,open_miss_pts,dpi_pts,penalty_nullified_pts,penalty_truncated_pts,overturned_td_pts,replay_reversal_pts,down_short_pts,down_short_fumble_pts,td_transfer_pts,hail_mary_pts,fumble_pts,drop_plays,open_miss_plays,dpi_plays,penalty_nullified_plays,penalty_truncated_plays,overturned_td_plays,replay_reversal_plays,down_short_plays,down_short_fumble_plays,td_transfer_plays,hail_mary_plays,fumble_plays
2921,2026,11,2026_11_JAX_NYG,00-0036309,0.00,26.90,0.00,Darnell Mooney,WR,NYG,26.90,26.90,0.00,25.20,0.00,1.70,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,4.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2135,2026,8,2026_08_CLE_PIT,00-0041037,9.70,26.60,0.00,Denzel Boston,WR,CLE,36.30,36.30,0.00,20.50,0.00,0.00,0.0,0.0,0.0,6.10,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1829,2026,7,2026_07_DEN_ARI,00-0033119,16.48,26.48,0.00,Jacoby Brissett,QB,BAL,42.96,42.96,9.24,0.00,2.04,0.98,0.0,0.0,0.0,14.22,0.0,0.0,0.0,0.0,3.0,0.0,4.0,2.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0
1807,2026,7,2026_07_CLE_TEN,00-0041037,5.70,25.60,1.50,Denzel Boston,WR,CLE,31.30,32.80,1.50,25.60,0.00,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1449,2026,5,2026_05_NYG_WAS,00-0039337,29.10,24.20,0.60,Malik Nabers,WR,NYG,53.30,53.90,0.60,21.80,0.00,2.40,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
740,2026,3,2026_03_KC_MIA,00-0039880,11.10,23.20,0.00,Malik Washington,WR,MIA,34.30,34.30,0.00,14.80,0.00,8.40,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
984,2026,4,2026_04_DEN_SF,00-0031408,12.60,22.70,0.00,Mike Evans,WR,SF,35.30,35.30,0.00,16.60,0.00,0.00,0.0,0.0,0.0,6.10,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1914,2026,7,2026_07_LA_LV,00-0039738,17.10,22.70,0.00,Blake Corum,RB,LA,39.80,39.80,0.00,0.00,0.00,16.50,0.0,0.0,0.0,6.20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1906,2026,7,2026_07_LA_LV,00-0037291,3.40,22.20,0.00,Jalen Nailor,WR,LV,25.60,25.60,0.00,19.60,0.00,0.00,0.0,0.0,2.6,0.00,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1604,2026,6,2026_06_DAL_GB,00-0036264,20.24,22.00,5.32,Jordan Love,QB,GB,42.24,47.56,11.52,5.32,1.52,0.80,0.0,0.0,0.0,8.16,0.0,0.0,0.0,0.0,6.0,2.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0


In [5]:
# games played
appeared = pd.concat([official[["player_id", "game_id"]], pg[["player_id", "game_id"]]]).drop_duplicates()

## Ranking
Ranks are by season total, within each position. Ties are broken by actual points. Ties on actual points are broken alphabetically.

Each rank is shown next to points per game, and the change columns show how many spots a player would move.

In [6]:
TOTALS = ["actual", "luck", "self_inflicted", "luck_adjusted", "should_be"]

def add_ranks(df, group_cols):
    """Rank actual, luck-adjusted, and should-be points within each position and group."""
    df = df.copy()
    for col in ["actual", "luck_adjusted", "should_be"]:
        order = df.sort_values([col, "actual", "player_display_name"],
                               ascending = [False, False, True])
        order[f"{col}_rank"] = order.groupby(group_cols + ["position_group"]).cumcount() + 1
        df = df.join(order[[f"{col}_rank"]])
    df["luck_rank_change"] = df["actual_rank"] - df["luck_adjusted_rank"]
    df["should_be_rank_change"] = df["actual_rank"] - df["should_be_rank"]
    for col in ["actual_rank", "luck_adjusted_rank", "should_be_rank"]:
        df[col.replace("_rank", "_label")] = df["position_group"] + df[col].astype(str)
    return df

def season_table(games):
    """Season totals, games played, and per-game averages for a set of player-games."""
    totals = games.groupby(["player_id", "player_display_name", "position_group", "team"],
                           as_index=False)[TOTALS + BREAKDOWN].sum()
    gp = (appeared[appeared["game_id"].isin(games["game_id"])]
          .groupby("player_id").size().rename("games").reset_index())
    totals = totals.merge(gp, on="player_id", how="left")
    for col in TOTALS:
        totals[f"{col}_per_game"] = totals[col] / totals["games"]
    return totals

In [7]:
# season to date
season = add_ranks(season_table(pg), [])
season = season.sort_values(["position_group", "actual_rank"])
season[season["position_group"] == "WR"][["player_display_name", "team", "games", "actual",
                                          "luck", "self_inflicted", "luck_adjusted", "should_be",
                                          "actual_label", "luck_adjusted_label", "should_be_label",
                                          "luck_rank_change"]].head(20).round(1)

,player_display_name,team,games,actual,luck,self_inflicted,luck_adjusted,should_be,actual_label,luck_adjusted_label,should_be_label,luck_rank_change
260,Jaxon Smith-Njigba,SEA,10,263.2,16.1,19.3,279.3,298.6,WR1,WR2,WR2,-1
137,CeeDee Lamb,DAL,11,252.2,44.8,10.4,297.0,307.4,WR2,WR1,WR1,1
171,Ja'Marr Chase,CIN,10,201.9,21.6,2.5,223.5,226.0,WR3,WR4,WR5,-1
177,Amon-Ra St. Brown,DET,10,198.1,28.3,0.0,226.4,226.4,WR4,WR3,WR4,1
141,Tee Higgins,CIN,10,183.7,25.6,4.2,209.3,213.5,WR5,WR7,WR11,-2
244,Christian Watson,GB,10,183.6,14.9,16.3,198.5,214.8,WR6,WR12,WR9,-6
446,Matthew Golden,GB,10,175.9,42.1,36.6,218.0,254.6,WR7,WR5,WR3,2
311,Puka Nacua,LA,7,173.6,32.0,13.5,205.6,219.1,WR8,WR10,WR8,-2
197,George Pickens,DAL,10,171.6,34.3,14.9,205.9,220.8,WR9,WR9,WR7,0
195,Chris Olave,NO,10,166.6,41.2,6.3,207.8,214.1,WR10,WR8,WR10,2


In [8]:
weekly_ranks = add_ranks(pg, ["season", "week"])
weekly_ranks = weekly_ranks.sort_values(["week", "position_group", "actual_rank"])

In [9]:
# season as of each week
weeks = sorted(pg["week"].unique())
as_of = pd.concat([season_table(pg[pg["week"] <= w]).assign(through_week=w) for w in weeks],
                  ignore_index=True)
as_of = add_ranks(as_of, ["through_week"])
print(f"Weeks: {weeks[0]} to {weeks[-1]}   Rows: {len(as_of):,}")

Weeks: 1 to 11   Rows: 4,936


## First look
The biggest rank moves at each position: who would climb the most if they caught every catchable ball and never lost points to turned-over plays.

In [10]:
RELEVANT = {"QB": 24, "RB": 48, "WR": 60, "TE": 24}

cols = ["player_display_name", "team", "games", "actual_label", "luck_adjusted_label",
        "luck_rank_change", "luck", "self_inflicted"]
for pos, n in RELEVANT.items():
    pool = season[(season["position_group"] == pos) & (season["actual_rank"] <= n)]
    print(f"\n{pos} (top {n} by actual points): unluckiest")
    print(pool.sort_values("luck", ascending=False)[cols].head(5).round(1).to_string(index=False))
    print(f"{pos}: luckiest")
    print(pool.sort_values("luck")[cols].head(5).round(1).to_string(index=False))


QB (top 24 by actual points): unluckiest
player_display_name team  games actual_label luck_adjusted_label  luck_rank_change  luck  self_inflicted
       Dak Prescott  DAL     11          QB2                 QB1                 1  47.9            35.0
    Jacoby Brissett  BAL     10         QB21                QB14                 7  47.1             9.3
        Jordan Love   GB     10          QB7                 QB5                 2  43.1            36.9
   Matthew Stafford   LA     10         QB14                QB12                 2  37.4            31.2
             Bo Nix  DEN     10         QB18                QB16                 2  34.4            20.6
QB: luckiest
player_display_name team  games actual_label luck_adjusted_label  luck_rank_change  luck  self_inflicted
       Tyler Shough   NO     10         QB13                QB17                -4   1.5            29.8
         Joe Burrow  CIN     10          QB1                 QB3                -2   2.8            19.5


In [11]:
season.to_parquet(f"data/season_ranks_{CURRENT_SZN}.parquet", index=False)
weekly_ranks.to_parquet(f"data/weekly_ranks_{CURRENT_SZN}.parquet", index=False)
as_of.to_parquet(f"data/ranks_by_week_{CURRENT_SZN}.parquet", index=False)